In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
import ast
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [3]:
# 1. LOAD DATA


print("\n Loading data...")
train = pd.read_csv('/kaggle/input/smart-data/train.csv')
test = pd.read_csv('/kaggle/input/smart-data/test.csv')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Target distribution:")
print(train['soil_moisture'].describe())



 Loading data...
Train: (68034, 35)
Test:  (29158, 34)
Target distribution:
count    68034.000000
mean         0.163570
std          0.110131
min          0.000000
25%          0.078000
50%          0.141000
75%          0.231500
max          0.599500
Name: soil_moisture, dtype: float64


In [4]:
# 2. ROBUST TEMPORAL PARSING




def safe_parse(val):
    
    if pd.isna(val):
        return []
    if isinstance(val, (int, float)) and not np.isnan(val):
        return [float(val)]
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return [float(x) for x in parsed if not pd.isna(x)]
            return [float(parsed)]
        except:
            return []
    if isinstance(val, list):
        return [float(x) for x in val if not pd.isna(x)]
    return []

def extract_robust_features(df, col):

    if col not in df.columns:
        return df
    
    print(f"   Processing: {col}")
    parsed = df[col].apply(safe_parse)
    
    # Mean (most stable)
    df[f'{col}_mean'] = parsed.apply(lambda x: np.mean(x) if len(x) > 0 else np.nan)
    
    # Standard deviation
    df[f'{col}_std'] = parsed.apply(lambda x: np.std(x) if len(x) > 1 else 0)
    
    # Min and Max
    df[f'{col}_min'] = parsed.apply(lambda x: np.min(x) if len(x) > 0 else np.nan)
    df[f'{col}_max'] = parsed.apply(lambda x: np.max(x) if len(x) > 0 else np.nan)
    
    # Recent value (last)
    df[f'{col}_last'] = parsed.apply(lambda x: x[-1] if len(x) > 0 else np.nan)
    
    # Range
    df[f'{col}_range'] = df[f'{col}_max'] - df[f'{col}_min']
    
    return df

print("\n  Extracting temporal features...")

# All temporal columns
temporal_cols = [
    'daily_temperature_2m',
    'daily_dewpoint_temperature_2m', 
    'VH',
    'VV',
    'hourly_surface_pressure',
    'daily_total_evaporation',
    'hourly_dewpoint_temperature_2m',
    'hourly_temperature_2m',
    'angle',
    'daily_total_precipitation',
    'daily_temperature_2m_max',
    'daily_temperature_2m_min',
    'daily_surface_pressure',
    'hourly_total_precipitation',
    'hourly_total_evaporation'
]

for col in temporal_cols:
    train = extract_robust_features(train, col)
    test = extract_robust_features(test, col)

print("    Temporal extraction complete")



  Extracting temporal features...
   Processing: daily_temperature_2m
   Processing: daily_temperature_2m
   Processing: daily_dewpoint_temperature_2m
   Processing: daily_dewpoint_temperature_2m
   Processing: VH
   Processing: VH
   Processing: VV
   Processing: VV
   Processing: hourly_surface_pressure
   Processing: hourly_surface_pressure
   Processing: daily_total_evaporation
   Processing: daily_total_evaporation
   Processing: hourly_dewpoint_temperature_2m
   Processing: hourly_dewpoint_temperature_2m
   Processing: hourly_temperature_2m
   Processing: hourly_temperature_2m
   Processing: angle
   Processing: angle
   Processing: daily_total_precipitation
   Processing: daily_total_precipitation
   Processing: daily_temperature_2m_max
   Processing: daily_temperature_2m_max
   Processing: daily_temperature_2m_min
   Processing: daily_temperature_2m_min
   Processing: daily_surface_pressure
   Processing: daily_surface_pressure
   Processing: hourly_total_precipitation
   Proc

In [5]:
# 3. COMPREHENSIVE FEATURE ENGINEERING



def create_features(df):
    df = df.copy()
    
    print("\n  Creating features...")
    
    # === TEMPORAL ===
    print("   - Temporal features")
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['year'] = df['datetime'].dt.year
    df['month'] = df['datetime'].dt.month
    df['day'] = df['datetime'].dt.day
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['week'] = df['datetime'].dt.isocalendar().week.astype(int)
    df['quarter'] = df['datetime'].dt.quarter
    df['season'] = ((df['month'] % 12 + 3) // 3).astype(int)
    
    # Cyclical features
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['day_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
    
    # === SPECTRAL INDICES ===
    print("   - Spectral indices")
    eps = 1e-10
    
    # Vegetation indices
    df['NDVI'] = (df['B8'] - df['B4']) / (df['B8'] + df['B4'] + eps)
    df['GNDVI'] = (df['B8'] - df['B3']) / (df['B8'] + df['B3'] + eps)
    df['RENDVI'] = (df['B8'] - df['B5']) / (df['B8'] + df['B5'] + eps)
    
    # Water/Moisture indices
    df['NDWI'] = (df['B8'] - df['B11']) / (df['B8'] + df['B11'] + eps)
    df['NDMI'] = (df['B8A'] - df['B11']) / (df['B8A'] + df['B11'] + eps)
    df['MNDWI'] = (df['B3'] - df['B11']) / (df['B3'] + df['B11'] + eps)
    
    # Soil indices
    df['BSI'] = ((df['B11'] + df['B4']) - (df['B8'] + df['B2'])) / ((df['B11'] + df['B4']) + (df['B8'] + df['B2']) + eps)
    
    # Enhanced vegetation
    df['EVI'] = 2.5 * ((df['B8'] - df['B4']) / (df['B8'] + 6*df['B4'] - 7.5*df['B2'] + 1))
    df['SAVI'] = 1.5 * ((df['B8'] - df['B4']) / (df['B8'] + df['B4'] + 0.5))
    
    # Band ratios
    df['B8_B4'] = df['B8'] / (df['B4'] + eps)
    df['B11_B8'] = df['B11'] / (df['B8'] + eps)
    df['B12_B8'] = df['B12'] / (df['B8'] + eps)
    df['B8_B3'] = df['B8'] / (df['B3'] + eps)
    
    # === RADAR ===
    print("   - Radar features")
    if 'VH_mean' in df.columns and 'VV_mean' in df.columns:
        df['VV_VH_ratio'] = df['VV_mean'] / (df['VH_mean'] + eps)
        df['VV_VH_diff'] = df['VV_mean'] - df['VH_mean']
        df['VV_VH_sum'] = df['VV_mean'] + df['VH_mean']
        df['RVI'] = 4 * df['VH_mean'] / (df['VV_mean'] + df['VH_mean'] + eps)
        
        # Polarization features
        df['pol_ratio'] = (df['VV_mean'] - df['VH_mean']) / (df['VV_mean'] + df['VH_mean'] + eps)
        
        if 'angle_mean' in df.columns:
            df['VV_normalized'] = df['VV_mean'] / (np.cos(np.radians(df['angle_mean'])) + eps)
            df['VH_normalized'] = df['VH_mean'] / (np.cos(np.radians(df['angle_mean'])) + eps)
    
    # === SOIL TEXTURE ===
    print("   - Soil features")
    df['sand_clay_ratio'] = df['sand'] / (df['clay'] + eps)
    df['silt_clay_ratio'] = df['silt'] / (df['clay'] + eps)
    df['sand_silt_ratio'] = df['sand'] / (df['silt'] + eps)
    
    # Water holding capacity (physical property)
    df['WHC'] = df['clay'] * 0.6 + df['silt'] * 0.35 + df['sand'] * 0.05
    df['porosity'] = 1 - (df['sand'] * 0.015 + df['silt'] * 0.012 + df['clay'] * 0.010)
    
    # Texture categories
    df['is_sandy'] = (df['sand'] > 70).astype(int)
    df['is_clayey'] = (df['clay'] > 40).astype(int)
    
    # === METEOROLOGICAL ===
    print("   - Meteorological features")
    
    # Temperature
    if 'hourly_temperature_2m_mean' in df.columns:
        df['temp_K'] = df['hourly_temperature_2m_mean']
        df['temp_C'] = df['temp_K'] - 273.15
        
        if 'hourly_dewpoint_temperature_2m_mean' in df.columns:
            # VPD - Vapor Pressure Deficit (critical)
            df['VPD'] = df['temp_K'] - df['hourly_dewpoint_temperature_2m_mean']
            df['VPD_squared'] = df['VPD'] ** 2
    
    # Temperature range
    if 'daily_temperature_2m_max_mean' in df.columns and 'daily_temperature_2m_min_mean' in df.columns:
        df['temp_range_daily'] = df['daily_temperature_2m_max_mean'] - df['daily_temperature_2m_min_mean']
    
    # Precipitation
    if 'daily_total_precipitation_mean' in df.columns:
        df['precip_mean'] = df['daily_total_precipitation_mean']
        df['precip_log'] = np.log1p(df['precip_mean'] * 1000)
        df['precip_sqrt'] = np.sqrt(df['precip_mean'] * 1000)
        df['has_precipitation'] = (df['precip_mean'] > 0.0001).astype(int)
    
    # Evaporation
    if 'daily_total_evaporation_mean' in df.columns:
        df['evap_mean'] = np.abs(df['daily_total_evaporation_mean'])
        df['evap_log'] = np.log1p(df['evap_mean'] * 1000)
    
    # WATER BALANCE (most critical)
    if 'precip_mean' in df.columns and 'evap_mean' in df.columns:
        df['water_balance'] = df['precip_mean'] - df['evap_mean']
        df['water_balance_abs'] = np.abs(df['water_balance'])
        df['PE_ratio'] = df['precip_mean'] / (df['evap_mean'] + eps)
        df['PE_diff'] = df['precip_mean'] - df['evap_mean']
    
    # === ELEVATION ===
    df['elevation_abs'] = np.abs(df['elevation'])
    df['elevation_log'] = np.log1p(df['elevation_abs'])
    df['is_high_elevation'] = (df['elevation_abs'] > 1000).astype(int)
    
    # === CLOUD ===
    df['cloud_pct'] = df['s2_cloud_percentage']
    df['is_cloudy'] = (df['cloud_pct'] > 30).astype(int)
    df['clear_sky_factor'] = 100 - df['cloud_pct']
    
    # === KEY INTERACTIONS ===
    print("   - Interaction features")
    
    # Soil × Moisture
    df['clay_WHC'] = df['clay'] * df['WHC']
    df['clay_precip'] = df['clay'] * df.get('precip_mean', 0)
    df['WHC_precip'] = df['WHC'] * df.get('precip_mean', 0)
    df['clay_water_balance'] = df['clay'] * df.get('water_balance', 0)
    df['WHC_water_balance'] = df['WHC'] * df.get('water_balance', 0)
    
    # Vegetation × Moisture
    df['NDVI_precip'] = df['NDVI'] * df.get('precip_mean', 0)
    df['NDWI_precip'] = df['NDWI'] * df.get('precip_mean', 0)
    df['NDMI_precip'] = df['NDMI'] * df.get('precip_mean', 0)
    df['NDVI_water_balance'] = df['NDVI'] * df.get('water_balance', 0)
    df['NDWI_water_balance'] = df['NDWI'] * df.get('water_balance', 0)
    
    # Vegetation × Soil
    df['NDVI_clay'] = df['NDVI'] * df['clay']
    df['NDVI_WHC'] = df['NDVI'] * df['WHC']
    df['NDWI_clay'] = df['NDWI'] * df['clay']
    
    # Temperature × Soil
    df['temp_clay'] = df.get('temp_C', 0) * df['clay']
    df['VPD_clay'] = df.get('VPD', 0) * df['clay']
    
    # Complex moisture proxies
    df['moisture_index_1'] = df['NDWI'] * df['WHC'] * df.get('water_balance', 0)
    df['moisture_index_2'] = df['NDMI'] * df['clay'] * df.get('precip_mean', 0)
    df['soil_moisture_proxy'] = (df.get('precip_mean', 0) * df['WHC']) / (df.get('evap_mean', 0.001) + eps)
    
    print("    Feature engineering complete")
    
    return df

train_fe = create_features(train)
test_fe = create_features(test)

print(f"\n Total columns: {len(train_fe.columns)}")


  Creating features...
   - Temporal features
   - Spectral indices
   - Radar features
   - Soil features
   - Meteorological features
   - Interaction features
    Feature engineering complete

  Creating features...
   - Temporal features
   - Spectral indices
   - Radar features
   - Soil features
   - Meteorological features
   - Interaction features
    Feature engineering complete

 Total columns: 200


In [6]:
# 4. TARGET ENCODING + LABEL ENCODING



print("\n  Encoding climate variable...")

# Target encoding with K-fold to avoid leakage
from sklearn.model_selection import KFold

def target_encode_kfold(train_df, test_df, col, target, n_splits=5):

    train_encoded = np.zeros(len(train_df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for train_idx, val_idx in kf.split(train_df):
        train_subset = train_df.iloc[train_idx]
        means = train_subset.groupby(col)[target].mean()
        train_encoded[val_idx] = train_df.iloc[val_idx][col].map(means)
    
    
    global_means = train_df.groupby(col)[target].mean()
    test_encoded = test_df[col].map(global_means)
    

    global_mean = train_df[target].mean()
    train_encoded = pd.Series(train_encoded).fillna(global_mean).values
    test_encoded = test_encoded.fillna(global_mean).values
    
    return train_encoded, test_encoded

train_fe['climate_target'], test_fe['climate_target'] = target_encode_kfold(
    train_fe, test_fe, 'climate', 'soil_moisture'
)


# Label encoding
le = LabelEncoder()
train_fe['climate_label'] = le.fit_transform(train_fe['climate'])
test_fe['climate_label'] = le.transform(test_fe['climate'])

print("    Climate encoding complete")


  Encoding climate variable...
    Climate encoding complete


In [7]:
# 5. DATA PREPARATION



print("\n  Preparing data...")

# Exclude columns
exclude = ['soil_moisture', 'datetime', 'climate'] + temporal_cols
exclude = [c for c in exclude if c in train_fe.columns]

feature_cols = [c for c in train_fe.columns if c not in exclude]

# Handle inf and nan
for df in [train_fe, test_fe]:
    df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    
# Fill with median
medians = train_fe[feature_cols].median()
train_fe[feature_cols] = train_fe[feature_cols].fillna(medians)
test_fe[feature_cols] = test_fe[feature_cols].fillna(medians)

X = train_fe[feature_cols].values
y = train_fe['soil_moisture'].values
X_test = test_fe[feature_cols].values

print(f"   Features: {len(feature_cols)}")
print(f"   Samples: {len(X)}")

# Stratified split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"   Train: {X_train.shape}")
print(f"   Val: {X_val.shape}")



  Preparing data...
   Features: 184
   Samples: 68034
   Train: (54427, 184)
   Val: (13607, 184)


In [8]:
# 6. MODEL TRAINING - LIGHTGBM



print("TRAINING LIGHTGBM")

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 100,
    'learning_rate': 0.03,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 5,
    'max_depth': -1,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val)

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=5000,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(150), lgb.log_evaluation(500)]
)

y_pred_lgb = lgb_model.predict(X_val)
r2_lgb = r2_score(y_val, y_pred_lgb)
rmse_lgb = np.sqrt(mean_squared_error(y_val, y_pred_lgb))

print(f"\n LightGBM - R²: {r2_lgb:.5f}, RMSE: {rmse_lgb:.5f}")


TRAINING LIGHTGBM
Training until validation scores don't improve for 150 rounds
[500]	valid_0's rmse: 0.0386664
[1000]	valid_0's rmse: 0.0359692
[1500]	valid_0's rmse: 0.0347813
[2000]	valid_0's rmse: 0.0341805
[2500]	valid_0's rmse: 0.0338535
[3000]	valid_0's rmse: 0.0336383
[3500]	valid_0's rmse: 0.033492
[4000]	valid_0's rmse: 0.0333941
[4500]	valid_0's rmse: 0.0333156
[5000]	valid_0's rmse: 0.0332659
Did not meet early stopping. Best iteration is:
[5000]	valid_0's rmse: 0.0332659

 LightGBM - R²: 0.90686, RMSE: 0.03327


In [9]:
# 7. MODEL TRAINING - XGBOOST



print("TRAINING XGBOOST")

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=150
)

xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=500)

y_pred_xgb = xgb_model.predict(X_val)
r2_xgb = r2_score(y_val, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))

print(f"\n XGBoost - R²: {r2_xgb:.5f}, RMSE: {rmse_xgb:.5f}")



TRAINING XGBOOST
[0]	validation_0-rmse:0.10712
[500]	validation_0-rmse:0.03879
[1000]	validation_0-rmse:0.03584
[1500]	validation_0-rmse:0.03475
[2000]	validation_0-rmse:0.03423
[2500]	validation_0-rmse:0.03398
[3000]	validation_0-rmse:0.03382
[3500]	validation_0-rmse:0.03372
[4000]	validation_0-rmse:0.03366
[4500]	validation_0-rmse:0.03362
[4999]	validation_0-rmse:0.03358

 XGBoost - R²: 0.90507, RMSE: 0.03358


In [10]:
# 8. MODEL TRAINING - CATBOOST



print("TRAINING CATBOOST")

cat_model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=500,
    early_stopping_rounds=150
)

cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))

y_pred_cat = cat_model.predict(X_val)
r2_cat = r2_score(y_val, y_pred_cat)
rmse_cat = np.sqrt(mean_squared_error(y_val, y_pred_cat))

print(f"\n CatBoost - R²: {r2_cat:.5f}, RMSE: {rmse_cat:.5f}")

TRAINING CATBOOST
0:	learn: 0.1089145	test: 0.1075341	best: 0.1075341 (0)	total: 181ms	remaining: 15m 5s
500:	learn: 0.0476128	test: 0.0492690	best: 0.0492690 (500)	total: 44.4s	remaining: 6m 38s
1000:	learn: 0.0392729	test: 0.0430995	best: 0.0430995 (1000)	total: 1m 28s	remaining: 5m 52s
1500:	learn: 0.0344810	test: 0.0402803	best: 0.0402803 (1500)	total: 2m 11s	remaining: 5m 6s
2000:	learn: 0.0309678	test: 0.0385002	best: 0.0385002 (2000)	total: 2m 54s	remaining: 4m 22s
2500:	learn: 0.0281708	test: 0.0372123	best: 0.0372123 (2500)	total: 3m 38s	remaining: 3m 38s
3000:	learn: 0.0258816	test: 0.0362709	best: 0.0362709 (3000)	total: 4m 22s	remaining: 2m 55s
3500:	learn: 0.0239468	test: 0.0355185	best: 0.0355185 (3500)	total: 5m 6s	remaining: 2m 11s
4000:	learn: 0.0223228	test: 0.0350383	best: 0.0350383 (4000)	total: 5m 50s	remaining: 1m 27s
4500:	learn: 0.0208291	test: 0.0345680	best: 0.0345680 (4500)	total: 6m 34s	remaining: 43.8s
4999:	learn: 0.0195075	test: 0.0342131	best: 0.0342131 

In [14]:
# 9. ENSEMBLE


print("CREATING ENSEMBLE")

best_r2 = 0
best_weights = None

for w1 in [0.3, 0.35, 0.4, 0.45, 0.5]:
    for w2 in [0.3, 0.35, 0.4, 0.45, 0.5]:
        w3 = 1 - w1 - w2
        if w3 < 0.1 or w3 > 0.5:
            continue
        
        pred = w1 * y_pred_lgb + w2 * y_pred_xgb + w3 * y_pred_cat
        r2 = r2_score(y_val, pred)
        
        if r2 > best_r2:
            best_r2 = r2
            best_weights = (w1, w2, w3)

w_lgb, w_xgb, w_cat = best_weights
y_pred_ensemble = w_lgb * y_pred_lgb + w_xgb * y_pred_xgb + w_cat * y_pred_cat
r2_ensemble = r2_score(y_val, y_pred_ensemble)
rmse_ensemble = np.sqrt(mean_squared_error(y_val, y_pred_ensemble))

print(f"\n Optimal Weights:")
print(f"   LightGBM: {w_lgb:.2f}")
print(f"   XGBoost:  {w_xgb:.2f}")
print(f"   CatBoost: {w_cat:.2f}")

print(f"\n ENSEMBLE PERFORMANCE:")
print(f"   R² Score:  {r2_ensemble:.5f}")
print(f"   RMSE:      {rmse_ensemble:.5f}")
print(f"   MAE:       {mean_absolute_error(y_val, y_pred_ensemble):.5f}")


CREATING ENSEMBLE

 Optimal Weights:
   LightGBM: 0.50
   XGBoost:  0.30
   CatBoost: 0.20

 ENSEMBLE PERFORMANCE:
   R² Score:  0.90859
   RMSE:      0.03295
   MAE:       0.02250


In [17]:
# 10. PREDICTIONS & SUBMISSION

print("GENERATING PREDICTIONS")

test_lgb = lgb_model.predict(X_test)
test_xgb = xgb_model.predict(X_test)
test_cat = cat_model.predict(X_test)
test_ensemble = w_lgb * test_lgb + w_xgb * test_xgb + w_cat * test_cat

# === FIX: Load sample_submission 
try:
    sample_sub = pd.read_csv('sample_submission.csv')
    print(f"   Sample submission has {len(sample_sub)} entries")
    print(f"   Our predictions have {len(test_ensemble)} entries")
    
    # Load original test.csv to get ID column
    test_original = pd.read_csv('/kaggle/input/smart25/test.csv')
    
    if 'ID' in test_original.columns:

        submission = pd.DataFrame({
            'ID': test_original['ID'].values,
            'TARGET': test_ensemble
        })
        
        # Verify and reorder to match sample_submission
        if len(submission) == len(sample_sub):
            # Reorder to match sample_submission order
            submission = submission.set_index('ID').loc[sample_sub['ID']].reset_index()
            print("   IDs matched and reordered according to sample_submission")
        else:
            print(f"    Warning: Length mismatch - Sample: {len(sample_sub)}, Ours: {len(submission)}")
    else:
        # Fallback: use IDs from sample_submission directly
        print("    No ID column in test.csv, using sample_submission IDs")
        submission = pd.DataFrame({
            'ID': sample_sub['ID'].values,
            'TARGET': test_ensemble
        })
    
    # Final verification
    print(f"\n Verification:")
    print(f"   Sample IDs count: {len(sample_sub)}")
    print(f"   Submission IDs count: {len(submission)}")
    print(f"   IDs match: {set(sample_sub['ID']) == set(submission['ID'])}")
    print(f"   Order matches: {list(sample_sub['ID']) == list(submission['ID'])}")
    
except FileNotFoundError:
    print("    sample_submission.csv not found, using fallback method")
    # Fallback method
    if 'ID' in test_fe.columns:
        ids = test_fe['ID'].values
    elif 'ID' in test.columns:
        ids = test['ID'].values
    else:
        ids = range(1, len(test_ensemble) + 1)
    
    submission = pd.DataFrame({
        'ID': ids,
        'TARGET': test_ensemble
    })

# Save submission
submission.to_csv('submission_file.csv', index=False)

print(f"\n Submission saved: submission_file.csv")
print(f"   Predictions: {len(submission)}")
print(f"   Range: [{test_ensemble.min():.4f}, {test_ensemble.max():.4f}]")
print(f"   Mean: {test_ensemble.mean():.4f}")

# Show first few rows
print(f"\n First 10 rows of submission:")
print(submission.head(10))

print(f" FINAL VALIDATION R²: {r2_ensemble:.5f}")


GENERATING PREDICTIONS
    sample_submission.csv not found, using fallback method

 Submission saved: submission_file.csv
   Predictions: 29158
   Range: [-0.0133, 0.5893]
   Mean: 0.1636

 First 10 rows of submission:
   ID    TARGET
0   1  0.261899
1   2  0.085092
2   3  0.228474
3   4  0.098412
4   5  0.092534
5   6  0.060066
6   7  0.029776
7   8  0.156785
8   9  0.348648
9  10  0.106872
 FINAL VALIDATION R²: 0.90859
